<a href="https://colab.research.google.com/github/jetisaplane1/AI-for-Business-BUS4-118S/blob/dev/Exercise1_PromptChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Exercise 1 — Prompt Chaining

!pip -q install --upgrade openai

import os, json, getpass
from openai import OpenAI

# ---- 1) Securely set API key in Colab session (do NOT hardcode in notebook) ----
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OPENAI_API_KEY: ")

# Create ONE client (do not overwrite it later)
client = OpenAI()  # SDK reads OPENAI_API_KEY automatically :contentReference[oaicite:3]{index=3}

def call_llm(instructions: str, user_input: str, model: str = "gpt-5.2") -> str:
    resp = client.responses.create(
        model=model,
        instructions=instructions,
        input=user_input,
    )
    return resp.output_text

def strict_json_loads(text: str) -> dict:
    """
    Attempts to parse JSON. If model returns extra text, tries to extract the first {...} block.
    (Helps you still get a successful run for your assignment.)
    """
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Try to extract a JSON object from the text
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = text[start:end+1]
            return json.loads(candidate)
        raise

customer_message = "I can’t log in since yesterday. It says 'invalid session'. My email is jet@example.com."

# STEP 1 — classify (JSON only)
instr1 = (
    "You are a customer support triage assistant. Output ONLY valid JSON (no code fences, no extra text). "
    "Classify the issue into one of: billing, login, shipping, refund, product_defect, other. "
    "Extract entities if present: order_id, email, product, error_message. "
    "Return confidence 0–1 and a short reason (max 20 words)."
)

out1_raw = call_llm(instr1, f"Customer message: {customer_message}")
print("STEP 1 RAW:\n", out1_raw)

step1 = strict_json_loads(out1_raw)
print("\nSTEP 1 PARSED JSON:\n", step1)

# STEP 2 — ask missing info questions
instr2 = (
    "You are a helpful support agent. Use the Step1 JSON. "
    "Ask at most 3 questions, only for missing info required to solve the category. "
    "Tone: calm, concise, professional. Output as a numbered list."
)

out2 = call_llm(
    instr2,
    f"Step1 JSON: {json.dumps(step1)}\nGoal: Ask only the missing questions needed to resolve this issue."
)
print("\nSTEP 2 QUESTIONS:\n", out2)

# Simulate customer answers (in a real flow, you'd capture user response)
customer_answers = "I’m using Chrome on Windows. Clearing cookies didn’t help. I can log in on my phone though."

# STEP 3 — solution
instr3 = (
    "Use Step 1 category + customer answers. Provide: "
    "(1) short diagnosis, (2) step-by-step fix (max 6 steps), (3) what to do if it fails. "
    "Do NOT request passwords, SSNs, or full card numbers. Output in Markdown with headings."
)

out3 = call_llm(
    instr3,
    f"Step1 JSON: {json.dumps(step1)}\nCustomer answers: {customer_answers}"
)
print("\nSTEP 3 SOLUTION:\n", out3)

# STEP 4 — escalation
instr4 = (
    "Decide whether to escalate to a human. Escalate if: confidence < 0.6 OR missing critical info "
    "after questions OR user is angry/threatening chargeback OR potential fraud. "
    "Output ONLY valid JSON with keys: escalate (bool), reason (string), handoff_summary (string, max 80 words)."
)

out4_raw = call_llm(
    instr4,
    f"Step1 JSON: {json.dumps(step1)}\nCustomer answers: {customer_answers}\nProposed solution: {out3}"
)
print("\nSTEP 4 RAW:\n", out4_raw)

step4 = strict_json_loads(out4_raw)
print("\nSTEP 4 PARSED JSON:\n", step4)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable